# AL Benchmark (S0-S3), Kaggle P100

In [ ]:
!pip install -q ultralytics==8.4.51

In [ ]:
CODE_DIR        = "/kaggle/input/ttcs-code"            # thư mục code
DATA_YAML       = "/kaggle/input/ttcs-export/export/data.yaml"  # file data.yaml
RESUME_DIR      = ""    # output session trước, rỗng là chạy mới
ONLY_STRATEGIES = []    # ví dụ ["random"], [] = chạy tất: random uncertainty coreset ppal
ONLY_SEEDS      = []    # ví dụ [13], [] = tất cả seeds: 13 42 1337
RUN_REPORT      = True  # chạy báo cáo cuối

In [ ]:
import os, sys, shutil
from pathlib import Path
import yaml

WORK = "/kaggle/working"
os.chdir(WORK)
sys.path.insert(0, CODE_DIR)

import torch
assert torch.cuda.is_available(), "Bật accelerator GPU P100 trong Settings."
print("GPU:", torch.cuda.get_device_name(0))
assert Path(CODE_DIR, "albench").exists(), f"albench/ không tồn tại: {CODE_DIR}"
assert Path(DATA_YAML).exists(), f"DATA_YAML không tồn tại: {DATA_YAML}"

_d = yaml.safe_load(Path(DATA_YAML).read_text())
_train = Path(_d.get("path", str(Path(DATA_YAML).parent))) / _d["train"]
_n = sum(p.suffix.lower() in {".jpg", ".jpeg", ".png"} for p in _train.glob("*")) if _train.exists() else 0
assert _n > 0, f"Không có ảnh train dưới {_train}"
print("ảnh train:", _n)

if RESUME_DIR:
    for sub in ("state", "reports", "splits"):
        s = Path(RESUME_DIR) / sub
        if s.exists():
            shutil.copytree(s, Path(WORK) / sub, dirs_exist_ok=True)
    print("resumed từ", RESUME_DIR)

In [ ]:
from albench.config import load_config, load_seeds
from albench.al import loop

cfg = load_config(Path(CODE_DIR) / "configs" / "benchmark.yaml")
cfg["al"]["cache"]       = False
cfg["baseline"]["cache"] = False
seeds      = ONLY_SEEDS      or load_seeds(Path(CODE_DIR) / "configs" / "seeds.yaml")[: cfg["al"]["seeds_n"]]
strategies = ONLY_STRATEGIES or cfg["al"]["strategies"]
print("strategies:", strategies, "| seeds:", seeds)

failures = []
for strat in strategies:
    for sd in seeds:
        print(f"{strat} seed={sd}", flush=True)
        try:
            loop.run(strat, sd, cfg, DATA_YAML)
        except Exception as e:
            print("FAILED:", strat, sd, repr(e))
            failures.append((strat, sd, repr(e)))
        torch.cuda.empty_cache()
print("failures:", failures)

In [ ]:
if RUN_REPORT:
    import subprocess
    rc = subprocess.run([
        sys.executable,
        str(Path(CODE_DIR) / "scripts" / "11_al_report.py"),
        "--config", str(Path(CODE_DIR) / "configs" / "benchmark.yaml"),
        "--data",   DATA_YAML,
        "--out",    "reports/al",
    ]).returncode
    print("report rc:", rc)
print("Xong. Nhấn Save Version.")

## Resume

- Save Version session hiện tại.
- Session mới: Add data, đặt RESUME_DIR là đường dẫn output session trước.
- Run All. Round đã xong tự bỏ qua.